In [ ]:
import pandas as pd
import numpy as np
from pandas import Series, DataFrame
from matplotlib import font_manager, rc
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
from matplotlib import cm
import matplotlib.font_manager as fm
plt.rc('font', family=fm.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()) # for Windows OS user
import math
%matplotlib inline
import seaborn as sns
import klib
import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectPercentile
from sklearn.model_selection import cross_val_score
from tqdm import tqdm
from sklearn.metrics import mean_squared_error
import sys, warnings
if not sys.warnoptions: warnings.simplefilter("ignore")
    

from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import AdaBoostRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import VotingRegressor

from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedKFold

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.base import ClassifierMixin
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

import os
import time
import random
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image
import pickle
from tqdm import tqdm
import platform
from itertools import combinations
from scipy.stats.mstats import gmean


import datetime
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.decomposition import PCA

In [ ]:
df_train = pd.read_csv('X_train.csv', encoding='cp949')
df_test = pd.read_csv('X_test.csv', encoding='cp949')
y_train = pd.read_csv('y_train.csv')
IDtest = df_test.custid.unique()
IDtrain = df_train.custid.unique()

train_id = df_train['custid']
test_id = df_test['custid']

df_all = pd.concat([df_train, df_test])

df_all.head()

In [ ]:
features = pd.DataFrame({'custid': df_all.custid.unique()})

### PCA

In [ ]:
# 차원축소 매소드 
from sklearn.decomposition import PCA

def dummy_to_pca(tr, column_name:str, features) :
    max_seq = 300
    max_d = 15
    col_count = tr.groupby(column_name)[column_name].count()
    if len(col_count) > max_seq:
        tops = col_count.sort_values(ascending=False)[0:max_seq].index
        f =tr.loc[tr[column_name].isin(tops)][['custid', column_name]]
    else:
        tops = col_count.index
        f =tr[['custid', column_name]]
    f = pd.get_dummies(f, columns=[column_name])  # This method performs One-hot-encoding
    f = f.groupby('custid').mean()
    if len(tops) < max_d:
        max_d = len(tops)
    pca = PCA(n_components=max_d)
    pca.fit(f)
    cumsum = np.cumsum(pca.explained_variance_ratio_) #분산의 설명량을 누적합
    #print(cumsum)
    num_d = np.argmax(cumsum >= 0.99) + 1 # 분산의 설명량이 99%이상 되는 차원의 수
    if num_d == 1:
        num_d = max_d
    pca = PCA(n_components=num_d)    
    result = pca.fit_transform(f)
    result = pd.DataFrame(result)
    result.columns = [column_name + '_' + str(column) for column in result.columns]
    result.index = f.index
    return result.reset_index()

#f = dummy_to_pca(tr, 'team_nm', f)
#f

### Make features

In [ ]:
df_all['date'] = df_all['sales_month']*100+df_all['sales_day']

In [ ]:
df_all['refund_tot'] = np.where(df_all['tot_amt']<0, (-1)*df_all['tot_amt'],0)



In [ ]:
f = df_all.groupby('custid').agg({
    'tot_amt': [('총구매액', 'sum'),('구매건수', 'size'),('평균구매가격', 'mean'),('최대구매액', 'max')],
    'dis_amt': [('dis_sum', 'sum'),('dis_mean', 'mean')],
    'net_amt': [('net_sum', 'sum'),('net_mean', 'mean')],
    'inst_mon': [('평균할부개월수', 'mean'), ('최대할부개월수', 'max')],
    'brd_nm': [('구매상품다양성비', lambda x: x.nunique()/x.count())],
    'str_nm': [('지점별다양성비', lambda x: x.nunique()/x.count())],
    'corner_nm': [('코너별다양성비', lambda x: x.nunique()/x.count())],
    'pc_nm': [('상품군별다양성비', lambda x: x.nunique()/x.count())],
    'part_nm': [('상품파트별다양성비', lambda x: x.nunique()/x.count())],
    'team_nm': [('상품팀별다양성비', lambda x: x.nunique()/x.count())],
    'buyer_nm': [('바이어별다양성비', lambda x: x.nunique()/x.count())],
    'import_flg': [('수입상품_구매비율', "mean"), ('수입상품_최대', 'max'), ('수입상품_최소', 'min')],
    'inst_fee' : [('무이자_구매비율', "mean"), ('무이자_최대', 'max'), ('무이자_최소', 'min')],
    'sales_month': [
        ('봄-구매비율', lambda x: np.mean( x.isin([15,16,5]))),
        ('여름-구매비율', lambda x: np.mean( x.isin([6,7,8]))),
        ('가을-구매비율', lambda x: np.mean( x.isin([9,10,11]))),
        ('겨울-구매비율', lambda x: np.mean( x.isin([13,14,12]))),
    ],
    'sales_dayofweek': [
        ('주말방문비율', lambda x: np.mean(x.isin(['토','일']))),
    ],
    'date': [
        ('내점일수',lambda x: x.nunique()),
        ('내점비율',lambda x: x.nunique()/x.count()),
    ],
    'sales_time': [('밤구입비율', lambda x: np.count_nonzero([(x>1800)|(x<900)])/ x.count())],
    }).reset_index()
features = features.merge(f, on='custid'); features




### PCA

In [ ]:
df_all['month'] = df_all['sales_month'].astype(str)
df_all['week'] = df_all['sales_dayofweek'].astype(str)
df_all['time'] = (df_all['sales_time']/100).astype(int).astype(str)
df_all['inst_mon1'] = df_all['inst_mon'].astype(str)
df_all['inst_fee1'] = df_all['inst_fee'].astype(str)

In [ ]:
f = dummy_to_pca(df_all, 'brd_nm', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'corner_nm', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'pc_nm', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'part_nm', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'buyer_nm', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'team_nm', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'goodcd', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'str_nm', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'month', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'week', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'time', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'inst_mon1', features)
features = features.merge(f, how='left'); features
f = dummy_to_pca(df_all, 'inst_fee1', features)
features = features.merge(f, how='left'); features
features

In [ ]:
list(features.columns)[9]

In [ ]:
list(features.columns)[2]

In [ ]:
features['내점당구매액'] = features[list(features.columns)[1]]/features[list(features.columns)[29]]
features['내점당평균구매액'] = features[list(features.columns)[3]]/features[list(features.columns)[29]]
features['내점당구매건수'] = features[list(features.columns)[2]]/features[list(features.columns)[29]];features



In [ ]:
f = df_all.query("tot_amt < 0").groupby('custid')['tot_amt'].agg([
    ('환불금액', np.sum)
]).reset_index()
f['환불금액'] = f['환불금액']*(-1)
features = features.merge(f, how='left')
features['환불금액'] = features['환불금액'].fillna(0)

f = df_all.query("tot_amt < 0").groupby('custid')['tot_amt'].agg([
    ('환불건수', np.size)
]).reset_index()
features = features.merge(f, how='left')
features['환불건수'] = features['환불건수'].fillna(0)
features

In [ ]:
features['환불비율'] = features['환불건수']/features[list(features.columns)[2]]
features['내점당환불액'] = features['환불금액']/features[list(features.columns)[29]]
features['내점당환불건수'] = features['환불건수']/features[list(features.columns)[29]]
features['구매건수대비평균환불액'] = features['환불금액']/features[list(features.columns)[2]]
features['한달평균납부액'] = features[list(features.columns)[1]]/features[list(features.columns)[9]]
features['한달결제취소액'] = features['환불금액']/features[list(features.columns)[9]];features



In [ ]:
f = df_all.query("tot_amt > 0").groupby('custid')['tot_amt'].agg([
    ('최소구매액', np.min)
]).reset_index()

features = features.merge(f, how='left'); features

무이자

In [ ]:
f = pd.crosstab(df_all.custid,df_all.inst_fee)
f.columns = ['무이자','이자']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.inst_fee, values = df_all.tot_amt, aggfunc='sum').fillna(0)
f.columns = ['무이자합계금액','이자합계금액']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.inst_fee, values = df_all.dis_amt, aggfunc='sum').fillna(0)
f.columns = ['무이자합계할인금액','이자합계할인금액']
features = features.merge(f, on = 'custid'); features

수입상품분류

In [ ]:
f = pd.crosstab(df_all.custid,df_all.import_flg)
f.columns = ['국내상품','수입상품']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid,df_all.import_flg, values = df_all.tot_amt, aggfunc='sum').fillna(0)
f.columns = ['국내상품합계금액','수입상품합계금액']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid,df_all.import_flg, values = df_all.dis_amt, aggfunc='sum').fillna(0)
f.columns = ['국내상품합계할인금액','수입상품합계할인금액']
features = features.merge(f, on = 'custid'); features

할부개월분류

In [ ]:
f = pd.crosstab(df_all.custid,df_all.inst_mon).fillna(0)
f.columns = ['할부1개월','할부2개월','할부3개월','할부4개월','할부5개월','할부6개월','할부7개월','할부8개월','할부9개월','할부10개월','할부11개월','할부12개월']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.inst_mon, values = df_all.tot_amt, aggfunc='sum').fillna(0)
f.columns = ['할부1개월합계금액','할부2개월합계금액','할부3개월합계금액','할부4개월합계금액','할부5개월합계금액','할부6개월합계금액','할부7개월합계금액','할부8개월합계금액','할부9개월합계금액','할부10개월합계금액','할부11개월합계금액','할부12개월합계금액']
features = features.merge(f, on = 'custid'); features


In [ ]:
f = pd.crosstab(df_all.custid, df_all.inst_mon, values = df_all.dis_amt, aggfunc='sum').fillna(0)
f.columns = ['할부1개월합계할인금액','할부2개월합계할인금액','할부3개월합계할인금액','할부4개월합계할인금액','할부5개월합계할인금액','할부6개월합계할인금액','할부7개월합계할인금액','할부8개월합계할인금액','할부9개월합계할인금액','할부10개월합계할인금액','할부11개월합계할인금액','할부12개월합계할인금액']
features = features.merge(f, on = 'custid'); features


월별방문횟수

In [ ]:
f = pd.crosstab(df_all.custid, df_all.sales_month).fillna(0)
f.columns = ['5월','6월','7월','8월','9월','10월','11월','12월','1월','2월','3월','4월']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.sales_month, values = df_all.tot_amt, aggfunc='sum').fillna(0)
f.columns = ['5월합계금액','6월합계금액','7월합계금액','8월합계금액','9월합계금액','10월합계금액','11월합계금액','12월합계금액','1월합계금액','2월합계금액','3월합계금액','4월합계금액']
features = features.merge(f, on = 'custid'); features


In [ ]:
f = pd.crosstab(df_all.custid, df_all.sales_month, values = df_all.dis_amt, aggfunc='sum').fillna(0)
f.columns = ['5월합계할인금액','6월합계할인금액','7월합계할인금액','8월합계할인금액','9월합계할인금액','10월합계할인금액','11월합계할인금액','12월합계할인금액','1월합계할인금액','2월합계할인금액','3월합계할인금액','4월합계할인금액']
features = features.merge(f, on = 'custid'); features

요일별분류

In [ ]:
f = pd.crosstab(df_all.custid, df_all.sales_dayofweek).fillna(0)
f.columns = ['금','목','수','월','일','토','화']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.sales_dayofweek, values = df_all.tot_amt, aggfunc='sum').fillna(0)
f.columns = ['금요일금액합계','목요일금액합계','수요일금액합계','월요일금액합계','일요일금액합계','토요일금액합계','화요일금액합계']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.sales_dayofweek, values = df_all.dis_amt, aggfunc='sum').fillna(0)
f.columns = ['금요일할인금액합계','목요일할인금액합계','수요일할인금액합계','월요일할인금액합계','일요일할인금액합계','토요일할인금액합계','화요일할인금액합계']
features = features.merge(f, on = 'custid'); features

In [ ]:
df_all['time'] = df_all['sales_time']//100
df_all['time'].unique()
df_all['time'] = np.where(df_all['time']<10, 10, df_all['time'])

In [ ]:
f = pd.crosstab(df_all.custid, df_all.time).fillna(0)
f.columns = ['10시','11시','시12시','13시','14시','15시','16시','17시','18시','19시','20사','21시','22시']
features = features.merge(f, on = 'custid'); features


In [ ]:
f = pd.crosstab(df_all.custid, df_all.time, values = df_all.tot_amt, aggfunc='sum').fillna(0)
f.columns = ['10시금액합계','11시금액합계','12시금액합계','13시금액합계','14시금액합계','15시금액합계','16시금액합계','17시금액합계','18시금액합계','19시금액합계','20사금액합계','21시금액합계','22시금액합계']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.time, values = df_all.dis_amt, aggfunc='sum').fillna(0)
f.columns = ['10시할인금액합계','11시할인금액합계','12시할인금액합계','13시할인금액합계','14시할인금액합계','15시할인금액합계','16시할인금액합계','17시할인금액합계','18시할인금액합계','19시할인금액합계','20시할인금액합계','21시할인금액합계','22시할인금액합계']
features = features.merge(f, on = 'custid'); features


지점별방문횟수

In [ ]:
f = pd.crosstab(df_all.custid, df_all.str_nm).fillna(0)
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.str_nm, values = df_all.tot_amt, aggfunc='sum').fillna(0)
f.columns = ['무역점금액합계','본점금액합계','신촌점금액합계','천호점금액합계']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.str_nm, values = df_all.dis_amt, aggfunc='sum').fillna(0)
f.columns = ['무역점할인금액합계','본점할인금액합계','신촌점할인금액합계','천호점할인금액합계']
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.part_nm).fillna(0)
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.team_nm).fillna(0)
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.buyer_nm).fillna(0)
features = features.merge(f, on = 'custid'); features

분류별갯수

In [ ]:
f = df_all.groupby('custid').agg({
    'str_nm': [('str_num', lambda x: x.nunique())],
    'brd_nm': [('brd_num', lambda x: x.nunique())],
    'corner_nm': [('corner_num', lambda x: x.nunique())],
    'pc_nm': [('pc_num', lambda x: x.nunique())],    
    'part_nm': [('part_num', lambda x: x.nunique())],
    'team_nm': [('team_num', lambda x: x.nunique())],
    'buyer_nm': [('buyer_num', lambda x: x.nunique())],
})

f.columns = f.columns.droplevel()
f=f.reset_index()

features = features.merge(f, how='left'); features

In [ ]:
features['str_div'] = features['str_num']/df_all['str_nm'].nunique()
features['brd_div'] = features['brd_num']/df_all['brd_nm'].nunique()
features['corner_div'] = features['corner_num']/df_all['corner_nm'].nunique()
features['pc_div'] = features['pc_num']/df_all['pc_nm'].nunique()
features['part_div'] = features['part_num']/df_all['part_nm'].nunique()
features['team_div'] = features['team_num']/df_all['team_nm'].nunique()
features['buyer_div'] = features['buyer_num']/df_all['buyer_nm'].nunique()

features

In [ ]:
f = df_all.groupby('custid').agg({
    'str_nm': [('str_max', lambda x: x.value_counts().index[0])],
    'brd_nm': [('brd_max', lambda x: x.value_counts().index[0])],
    'corner_nm': [('corner_max', lambda x: x.value_counts().index[0])],
    'pc_nm': [('pc_max', lambda x: x.value_counts().index[0])],   
    'part_nm': [('part_max', lambda x: x.value_counts().index[0])],
    'team_nm': [('team_max', lambda x: x.value_counts().index[0])],
    'buyer_nm': [('buyer_max', lambda x: x.value_counts().index[0])],
    'goodcd': [('goodcd_max', lambda x: x.value_counts().index[0])],
})

f.columns = f.columns.droplevel()
f=f.reset_index()

#features = features.merge(f, how='left'); features
df_all = df_all.merge(f, how='left'); df_all

In [ ]:
### 주구매지점 구매액

f = df_all.query("str_nm == str_max").groupby('custid')['tot_amt'].agg([
    ('str_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left')

### 주브랜드 구매액

f = df_all.query("brd_nm == brd_max").groupby('custid')['tot_amt'].agg([
    ('brd_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left')

### 주코너 구매액

f = df_all.query("corner_nm == corner_max").groupby('custid')['tot_amt'].agg([
    ('corner_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left')

### 주상품군 구매액

f = df_all.query("pc_nm == pc_max").groupby('custid')['tot_amt'].agg([
    ('pc_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left')

### 주파트 구매액

f = df_all.query("part_nm == part_max").groupby('custid')['tot_amt'].agg([
    ('part_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left')

### 주관리팀 구매액

f = df_all.query("team_nm == team_max").groupby('custid')['tot_amt'].agg([
    ('team_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left')

### 주바이어 구매액

f = df_all.query("buyer_nm == buyer_max").groupby('custid')['tot_amt'].agg([
    ('buyer_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left')

### 주상품 구매액

f = df_all.query("goodcd == goodcd_max").groupby('custid')['tot_amt'].agg([
    ('goodcd_tot', np.sum)
]).reset_index()

features = features.merge(f, how='left');features

In [ ]:
#구매등급
f = df_all.groupby('custid')['tot_amt'].agg([
    ('총구매액',np.sum),
    ('구매횟수',np.size)]).reset_index()

for i, row in f.iterrows() :
    if f.at[i,'총구매액'] >= 6000000 and f.at[i,'구매횟수'] >= 12:
        f.at[i,'구매등급'] =  'A'
    elif 4500000 <= f.at[i,'총구매액'] < 6000000 and f.at[i,'구매횟수'] >= 12:
        f.at[i,'구매등급'] =  'B'
    elif 3000000 <= f.at[i,'총구매액'] < 4500000 and f.at[i,'구매횟수'] >= 12:
         f.at[i,'구매등급'] = 'C'
    elif 1500000 <= f.at[i,'총구매액'] < 3000000 and f.at[i,'구매횟수'] >= 12:
        f.at[i,'구매등급'] = 'D'
    else :
        f.at[i,'구매등급'] = 'E'
f=f[['custid','구매등급']]
df_all = df_all.merge(f, how='left');df_all

In [ ]:
f = pd.crosstab(df_all.custid, df_all.구매등급).fillna(0)
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.구매등급, values = df_all.tot_amt, aggfunc='sum').fillna(0)
features = features.merge(f, on = 'custid'); features

In [ ]:
f = pd.crosstab(df_all.custid, df_all.구매등급, values = df_all.dis_amt, aggfunc='sum').fillna(0)
features = features.merge(f, on = 'custid'); features

In [ ]:
x = df_all[df_all['import_flg'] == 1].groupby('custid').size() / df_all.groupby('custid').size()
f = x.reset_index().rename(columns={0: '수입상품_구매비율'}).fillna(0)
f.iloc[:,1] = (f.iloc[:,1]*100).apply(round, args=(1,))
features = features.merge(f, how='left'); features

In [ ]:
def f2(x):
    if x in ['월','화','수']:
        return('월화수_구매건수')
    elif x in ['목','금']:
        return('목금_구매건수')
    elif x in '토':
        return('토_구매건수')
    else :
        return('일_구매건수')    
    
df_all['요일2'] = df_all['sales_dayofweek'].apply(f2)
f = pd.pivot_table(df_all, index='custid', columns='요일2', values='tot_amt', 
                   aggfunc=np.size, fill_value=0).reset_index()
features = features.merge(f, how='left'); features

In [ ]:
def f1(x):
    if x in [14,15,16]:
        return('234월_구매건수')
    elif x in [5,6,7]:
        return('567월_구매건수')
    elif x in [8,9,10]:
        return('8910월_구매건수')
    else :
        return('11121월_구매건수')    
    
df_all['season2'] = df_all['sales_month'].apply(f1)
f = pd.pivot_table(df_all, index='custid', columns='season2', values='tot_amt', 
                   aggfunc=np.size, fill_value=0).reset_index()
features = features.merge(f, how='left'); features

In [ ]:
def f2(x):
    if 901 <= x < 1200 :
        return('12시 이전_구매건수')
    elif 1200 <= x < 1400 :
        return('12~2시_구매건수')
    elif 1400 <= x < 1600 :
        return('2~4시_구매건수')
    elif 1600 <= x < 1800 :
        return('4~6시_구매건수')
    else :
        return('6시이후_구매건수')  

df_all['timeslot2'] = df_all.sales_time.apply(f2)

f = pd.pivot_table(df_all, index='custid', columns='timeslot2', values='tot_amt',
                   aggfunc=np.size, fill_value=0).reset_index()
features = features.merge(f, how='left'); features

In [ ]:
f = df_all.groupby('custid')['sales_time'].agg([('평균구매시간', 'mean')]).reset_index()
f.iloc[:,1] = f.iloc[:,1].apply(round, args=(1,))
features = features.merge(f, how='left'); features

In [ ]:
f = df_all.groupby('custid')['refund_tot'].agg([('환불액평균', 'mean')]).reset_index()
f.iloc[:,1] = f.iloc[:,1].apply(round, args=(1,))
features = features.merge(f, how='left'); features

In [ ]:
features = features.drop('custid', axis = 1)

### Data Cleansing & Feature Engineering

In [ ]:
# 범주형 변수와 수치형 변수를 분리

cat_features = features.select_dtypes(include=['object']).columns.to_list()
num_features = features.select_dtypes(exclude='object').columns.to_list()

In [ ]:
# 결측값 처리: 범주형이냐 수치형이냐에 따라 다르게 처리
from sklearn.impute import SimpleImputer 

if len(num_features) > 0: 
    features[num_features] = SimpleImputer(strategy='constant', fill_value=0).fit_transform(features[num_features])
if len(cat_features) > 0:  
    features[cat_features] = SimpleImputer(strategy="most_frequent").fit_transform(features[cat_features])

features

In [ ]:
# 수치형 변수에 대해 이상치(outlier)를 처리

features[num_features] = features[num_features].apply(lambda x: x.clip(x.quantile(.05), x.quantile(.95)), axis=0)

In [ ]:
# 수치형 변수를 정규분포에 가깝게 만들기 + 표준화
from sklearn.preprocessing import PowerTransformer, StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer

# mmscaler = MinMaxScaler()
# rsscaler = RobustScaler()
#features[num_features] = mmscaler.fit_transform(features[num_features])
#features[num_features] = np.log1p(features[num_features])
#features[num_features] = PowerTransformer(standardize=True).fit_transform(features[num_features])
#features[num_features] = rsscaler.fit_transform(features[num_features])

# features

In [ ]:
scaler = StandardScaler()

features[num_features] = scaler.fit_transform(features[num_features])
features

In [ ]:
# 범주형 변수에 One-Hot-Encoding 후 수치형 변수와 병합

if len(cat_features) > 0:
    features = pd.concat([features[num_features], pd.get_dummies(features[cat_features])], axis=1)
else:
    features = features[num_features]

features

In [ ]:
features.insert(0,'custid',df_all['custid'].unique())
features

In [ ]:
idx_train = pd.DataFrame({'custid': df_train['custid'].unique()})
X_train = pd.merge(idx_train, features, how='left', on='custid')

idx_test = pd.DataFrame({'custid': df_test['custid'].unique()})
X_test = pd.merge(idx_test, features, how='left', on='custid')

In [ ]:
X_train

### W2V(Corner_nm)

In [ ]:
level = 'corner_nm' # 상품 분류 수준

# W2V 학습을 하기에는 데이터(즉 corpus)가 부족하여 
# 고객별로 구매한 상품 목록으로부터 20배 oversampling을 수행
sentences = []
df_all = pd.concat([df_train, df_test])
for id in df_all.custid.unique():
#    uw = df_all.query('custid == @id')[level].unique()
#    bs = np.array([])
#    for j in range(20):
#        bs = np.append(bs, np.random.choice(uw, len(uw), replace=False))
#    sentences.append(list(bs))
    sentences.append(list(df_all.query('custid == @id')[level].values))

In [ ]:
max_features = 300 # 문자 벡터 차원 수
min_word_count = 1 # 최소 문자 수
num_workers = 4 # 병렬 처리 스레드 수
context = 3 # 문자열 창 크기
downsampling = 1e-3 # 문자 빈도수 Downsample

from gensim.models import word2vec

# 모델 학습
model = word2vec.Word2Vec(sentences, 
                          workers=num_workers, 
                          vector_size=max_features, 
                          min_count=min_word_count,
                          window=context,
                          sample=downsampling)
# 필요없는 메모리 unload
model.init_sims(replace=True)

In [ ]:
def age_vec():
    truth = pd.read_csv('y_train.csv')

    sentences = []
    df_all = df_train
    for id in df_all.custid.unique():
        x = df_all.query('custid == @id')[level].unique()
        y = np.where(truth.query('custid == @id').group.str[1:].astype(int))
        for j in range(20):
            y = np.append(y, np.random.choice(x, len(x), replace=False))
        sentences.append(list(y))

In [ ]:
# 고객별로 구매한 상품의 평균벡터를 feature로 사용한다.
features = []
for id in df_train.custid.unique():
    features.append(df_all.query('custid == @id')[level] \
                              .apply(lambda x: model.wv[x]).mean())
X_train_corner = np.array(features)

features = []
for id in df_test.custid.unique():
    features.append(df_all.query('custid == @id')[level] \
                              .apply(lambda x: model.wv[x]).mean())
X_test_corner = np.array(features)

In [ ]:
X_train_corner = pd.DataFrame(X_train_corner)
X_test_corner = pd.DataFrame(X_test_corner)

In [ ]:
idx_train = pd.DataFrame({'custid': df_train['custid'].unique()})
X_train_corner = pd.DataFrame(X_train_corner)
X_train_corner.insert(0,'custid',idx_train)

idx_test = pd.DataFrame({'custid': df_test['custid'].unique()})
X_test_corner = pd.DataFrame(X_test_corner)
X_test_corner.insert(0,'custid',idx_test)

In [ ]:
X_train = pd.merge(X_train, X_train_corner, on='custid')
X_test = pd.merge(X_test, X_test_corner, on='custid')

In [ ]:
X_train

### W2V (brd_nm)

In [ ]:
### Make corpus
level = 'brd_nm' # 상품 분류 수준

# W2V 학습을 하기에는 데이터(즉 corpus)가 부족하여 
# 고객별로 구매한 상품 목록으로부터 20배 oversampling을 수행
sentences = []
df_all = pd.concat([df_train, df_test])
for id in df_all.custid.unique():
#    uw = df_all.query('custid == @id')[level].unique()
#    bs = np.array([])
#    for j in range(20):
#        bs = np.append(bs, np.random.choice(uw, len(uw), replace=False))
#    sentences.append(list(bs))
    sentences.append(list(df_all.query('custid == @id')[level].values))

### Training the Word2Vec model
max_features = 300 # 문자 벡터 차원 수
min_word_count = 1 # 최소 문자 수
num_workers = 4 # 병렬 처리 스레드 수
context = 3 # 문자열 창 크기
downsampling = 1e-3 # 문자 빈도수 Downsample

from gensim.models import word2vec

# 모델 학습
model = word2vec.Word2Vec(sentences, 
                          workers=num_workers, 
                          vector_size=max_features, 
                          min_count=min_word_count,
                          window=context,
                          sample=downsampling)
# 필요없는 메모리 unload
model.init_sims(replace=True)

In [ ]:
def age_vec():
    truth = pd.read_csv('y_train.csv')

    sentences = []
    df_all = df_train
    for id in df_all.custid.unique():
        x = df_all.query('custid == @id')[level].unique()
        y = np.where(truth.query('custid == @id').group.str[1:].astype(int))
        for j in range(20):
            y = np.append(y, np.random.choice(x, len(x), replace=False))
        sentences.append(list(y))

In [ ]:
# 고객별로 구매한 상품의 평균벡터를 feature로 사용한다.
features = []
for id in df_train.custid.unique():
    features.append(df_all.query('custid == @id')[level] \
                              .apply(lambda x: model.wv[x]).mean())
X_train_brd = np.array(features)

features = []
for id in df_test.custid.unique():
    features.append(df_all.query('custid == @id')[level] \
                              .apply(lambda x: model.wv[x]).mean())
X_test_brd = np.array(features)

In [ ]:
X_train_brd = pd.DataFrame(X_train_brd)
X_test_brd = pd.DataFrame(X_test_brd)

In [ ]:
idx_train = pd.DataFrame({'custid': df_train['custid'].unique()})
X_train_brd = pd.DataFrame(X_train_brd)
X_train_brd.insert(0,'custid',idx_train)

idx_test = pd.DataFrame({'custid': df_test['custid'].unique()})
X_test_brd = pd.DataFrame(X_test_brd)
X_test_brd.insert(0,'custid',idx_test)

In [ ]:
X_train = pd.merge(X_train, X_train_brd, on='custid')
X_test = pd.merge(X_test, X_test_brd, on='custid')

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
X_train.to_csv('3.3rd_feature_0612.csv' , index = False )
X_test.to_csv('3.3rd_feature_te_0612.csv', index = False )